# Session 3, Module 02: Custom Exceptions


This module covers:
- Creating custom exception classes
- Exception hierarchy design
- Adding context to exceptions
- When to use custom vs built-in exceptions

Data Engineering Context:
Custom exceptions make error handling clearer in ETL pipelines
and help distinguish between different failure modes.


## Why Custom Exceptions?


In [2]:
print("=== Why Custom Exceptions? ===")

=== Why Custom Exceptions? ===


Built-in exceptions are generic
Custom exceptions:
1. Provide clearer error messages
2. Allow catching specific error types
3. Can carry additional context
4. Make code more self-documenting

In [3]:
print("""
Benefits of custom exceptions:
- Clearer error messages for debugging
- Catch specific errors, not all errors
- Include relevant context (record ID, file path, etc.)
- Self-documenting: PipelineError vs generic Exception
""")


Benefits of custom exceptions:
- Clearer error messages for debugging
- Catch specific errors, not all errors
- Include relevant context (record ID, file path, etc.)
- Self-documenting: PipelineError vs generic Exception



## Basic Custom Exception


In [4]:
print("\n=== Basic Custom Exception ===")


# Simplest form: just inherit from Exception
class PipelineError(Exception):
    """Base exception for all pipeline errors."""
    pass


# Usage
try:
    raise PipelineError("Pipeline failed to complete")
except PipelineError as e:
    print(f"Caught PipelineError: {e}")  # OUTPUT: Caught PipelineError: Pipeline failed to complete


=== Basic Custom Exception ===
Caught PipelineError: Pipeline failed to complete


## Custom Exception With Context


In [5]:
print("\n=== Custom Exception with Context ===")


class ExtractionError(Exception):
    """Error during data extraction phase."""

    def __init__(self, message: str, source: str = None, records_read: int = 0):
        """
        Initialize ExtractionError with context.

        Args:
            message: Error description
            source: Data source (file path, URL, table name)
            records_read: Number of records successfully read before error
        """
        super().__init__(message)
        self.source = source
        self.records_read = records_read

    def __str__(self):
        """Return formatted error message with context."""
        base = super().__str__()
        if self.source:
            base = f"{base} (source: {self.source})"
        if self.records_read > 0:
            base = f"{base} [read {self.records_read} records before failure]"
        return base


# Usage with context
try:
    raise ExtractionError(
        "Failed to parse CSV",
        source="/data/customers.csv",
        records_read=1523
    )
except ExtractionError as e:
    print(f"Error: {e}")
    print(f"  Source: {e.source}")
    print(f"  Records read: {e.records_read}")
    # OUTPUT:


=== Custom Exception with Context ===
Error: Failed to parse CSV (source: /data/customers.csv) [read 1523 records before failure]
  Source: /data/customers.csv
  Records read: 1523


Error: Failed to parse CSV (source: /data/customers.csv) [read 1523 records before failure]
Source: /data/customers.csv
Records read: 1523

## Exception Hierarchy


In [6]:
print("\n=== Exception Hierarchy ===")


=== Exception Hierarchy ===


Design a hierarchy for different error types
This allows catching at different levels of specificity

In [7]:
class DataPipelineError(Exception):
    """Base exception for all data pipeline errors."""

    def __init__(self, message: str, **context):
        super().__init__(message)
        self.context = context

    def __str__(self):
        base = super().__str__()
        if self.context:
            ctx = ", ".join(f"{k}={v}" for k, v in self.context.items())
            return f"{base} [{ctx}]"
        return base


class ExtractError(DataPipelineError):
    """Error during extraction phase."""
    pass


class TransformError(DataPipelineError):
    """Error during transformation phase."""
    pass


class LoadError(DataPipelineError):
    """Error during load phase."""
    pass


class ValidationError(TransformError):
    """Data validation failed during transformation."""

    def __init__(self, message: str, field: str = None, value=None, **context):
        super().__init__(message, field=field, value=value, **context)
        self.field = field
        self.value = value


class SchemaError(TransformError):
    """Schema mismatch during transformation."""

    def __init__(self, message: str, expected=None, actual=None, **context):
        super().__init__(message, expected=expected, actual=actual, **context)
        self.expected = expected
        self.actual = actual


# Demonstrate hierarchy
print("Exception Hierarchy:")
print(f"  DataPipelineError")
print(f"  ├── ExtractError")
print(f"  ├── TransformError")
print(f"  │   ├── ValidationError")
print(f"  │   └── SchemaError")
print(f"  └── LoadError")

# Catching at different levels
errors_to_test = [
    ValidationError("Invalid email", field="email", value="not-an-email"),
    SchemaError("Missing column", expected=["id", "name"], actual=["id"]),
    ExtractError("Connection timeout", source="api.example.com"),
    LoadError("Write failed", destination="warehouse"),
]

print("\nCatching at different levels:")

for error in errors_to_test:
    try:
        raise error
    except ValidationError as e:
        print(f"  ValidationError: {e}")
    except TransformError as e:
        print(f"  TransformError (general): {e}")
    except DataPipelineError as e:
        print(f"  DataPipelineError (base): {e}")

Exception Hierarchy:
  DataPipelineError
  ├── ExtractError
  ├── TransformError
  │   ├── ValidationError
  │   └── SchemaError
  └── LoadError

Catching at different levels:
  ValidationError: Invalid email [field=email, value=not-an-email]
  TransformError (general): Missing column [expected=['id', 'name'], actual=['id']]
  DataPipelineError (base): Connection timeout [source=api.example.com]
  DataPipelineError (base): Write failed [destination=warehouse]


## Re-Raising With Context


In [8]:
print("\n=== Re-raising with Context ===")


def process_record(record: dict, index: int) -> dict:
    """Process a single record, adding context on error."""
    if not record.get("id"):
        raise ValidationError(
            "Missing required field",
            field="id",
            record_index=index
        )
    return {"processed": True, **record}


def process_batch(records: list[dict]) -> list[dict]:
    """Process a batch of records."""
    results = []

    for i, record in enumerate(records):
        try:
            results.append(process_record(record, i))
        except ValidationError as e:
            # Re-raise with additional batch context
            raise ValidationError(
                str(e),
                field=e.field,
                record_index=i,
                batch_size=len(records)
            ) from e  # 'from e' chains the exceptions

    return results


# Test re-raising
test_records = [
    {"id": 1, "name": "Alice"},
    {"name": "Missing ID"},  # This will fail
]

try:
    process_batch(test_records)
except ValidationError as e:
    print(f"Validation failed: {e}")
    print(f"  Field: {e.field}")
    print(f"  Context: {e.context}")
    if e.__cause__:
        print(f"  Original error: {e.__cause__}")


=== Re-raising with Context ===
Validation failed: Missing required field [field=id, value=None, record_index=1] [field=id, value=None, record_index=1, batch_size=2]
  Field: id
  Context: {'field': 'id', 'value': None, 'record_index': 1, 'batch_size': 2}
  Original error: Missing required field [field=id, value=None, record_index=1]


## Exception With Recovery Suggestions


In [9]:
print("\n=== Exception with Recovery Suggestions ===")


class RecoverableError(DataPipelineError):
    """Error that includes recovery suggestions."""

    def __init__(self, message: str, suggestions: list[str] = None, **context):
        super().__init__(message, **context)
        self.suggestions = suggestions or []

    def get_help(self) -> str:
        """Return formatted help text."""
        if not self.suggestions:
            return "No recovery suggestions available."

        lines = ["Recovery suggestions:"]
        for i, suggestion in enumerate(self.suggestions, 1):
            lines.append(f"  {i}. {suggestion}")
        return "\n".join(lines)


class ConnectionError(RecoverableError):
    """Database connection failed."""

    def __init__(self, message: str, host: str = None, port: int = None, **context):
        suggestions = [
            "Check if the database server is running",
            "Verify network connectivity to the host",
            "Confirm credentials are correct",
            "Check firewall rules",
        ]
        super().__init__(message, suggestions=suggestions, host=host, port=port, **context)


# Usage
try:
    raise ConnectionError(
        "Connection refused",
        host="db.example.com",
        port=5432
    )
except ConnectionError as e:
    print(f"Error: {e}")
    print()
    print(e.get_help())


=== Exception with Recovery Suggestions ===
Error: Connection refused [host=db.example.com, port=5432]

Recovery suggestions:
  1. Check if the database server is running
  2. Verify network connectivity to the host
  3. Confirm credentials are correct
  4. Check firewall rules


## When To Use Custom Vs Built-In Exceptions


In [10]:
print("\n=== When to Use Custom vs Built-in ===")

print("""
USE BUILT-IN EXCEPTIONS when:
  - The error fits a standard category (ValueError, TypeError, etc.)
  - No additional context is needed
  - The error is a programming mistake, not a runtime issue

USE CUSTOM EXCEPTIONS when:
  - You need domain-specific error types (PipelineError, etc.)
  - Additional context is valuable (source file, record count, etc.)
  - You want to catch specific error categories
  - Recovery suggestions would help users

COMMON PATTERNS:
  - Base exception for your library/application
  - Subclasses for each error category
  - Include context as attributes
  - Implement __str__ for readable messages
""")

# Example of when to use built-in
def validate_age(age):
    """Use built-in ValueError for invalid argument."""
    if not isinstance(age, int):
        raise TypeError(f"age must be int, got {type(age).__name__}")
    if age < 0:
        raise ValueError(f"age must be non-negative, got {age}")
    return age


# Example of when to use custom
def validate_customer_record(record: dict):
    """Use custom ValidationError for domain validation."""
    if "customer_id" not in record:
        raise ValidationError(
            "Missing customer_id",
            field="customer_id",
            record=record
        )
    if record.get("age") and record["age"] < 0:
        raise ValidationError(
            "Invalid age value",
            field="age",
            value=record["age"],
            record_id=record.get("customer_id")
        )


=== When to Use Custom vs Built-in ===

USE BUILT-IN EXCEPTIONS when:
  - The error fits a standard category (ValueError, TypeError, etc.)
  - No additional context is needed
  - The error is a programming mistake, not a runtime issue

USE CUSTOM EXCEPTIONS when:
  - You need domain-specific error types (PipelineError, etc.)
  - Additional context is valuable (source file, record count, etc.)
  - You want to catch specific error categories
  - Recovery suggestions would help users

COMMON PATTERNS:
  - Base exception for your library/application
  - Subclasses for each error category
  - Include context as attributes
  - Implement __str__ for readable messages



## Practical Example: Etl Exception Hierarchy


In [11]:
print("\n=== Practical Example: ETL Exception Hierarchy ===")


# Complete exception hierarchy for an ETL system
class ETLError(Exception):
    """Base class for ETL errors."""

    error_code: str = "ETL000"

    def __init__(self, message: str, **context):
        super().__init__(message)
        self.context = context

    def to_dict(self) -> dict:
        """Convert error to dictionary for logging/API responses."""
        return {
            "error_code": self.error_code,
            "error_type": self.__class__.__name__,
            "message": str(self),
            "context": self.context,
        }


class SourceError(ETLError):
    """Error related to data source."""
    error_code = "ETL100"


class SourceNotFoundError(SourceError):
    """Source file or table does not exist."""
    error_code = "ETL101"


class SourceAccessError(SourceError):
    """Cannot access source (permissions, network, etc.)."""
    error_code = "ETL102"


class TransformationError(ETLError):
    """Error during data transformation."""
    error_code = "ETL200"


class DataQualityError(TransformationError):
    """Data quality check failed."""
    error_code = "ETL201"


class DestinationError(ETLError):
    """Error writing to destination."""
    error_code = "ETL300"


# Usage
errors = [
    SourceNotFoundError("File not found", path="/data/missing.csv"),
    DataQualityError("Null rate exceeds threshold", column="email", null_rate=0.15, threshold=0.05),
    DestinationError("Table locked", table="dim_customers"),
]

print("ETL Errors as dictionaries (for logging):")
for error in errors:
    error_dict = error.to_dict()
    print(f"\n{error_dict['error_code']}: {error_dict['message']}")
    print(f"  Context: {error_dict['context']}")


=== Practical Example: ETL Exception Hierarchy ===
ETL Errors as dictionaries (for logging):

ETL101: File not found
  Context: {'path': '/data/missing.csv'}

ETL201: Null rate exceeds threshold
  Context: {'column': 'email', 'null_rate': 0.15, 'threshold': 0.05}

ETL300: Table locked
  Context: {'table': 'dim_customers'}


## Summary


In [ ]:
print("\n=== Summary ===")
print("""
Custom Exceptions Best Practices:

1. HIERARCHY DESIGN:
   - Base exception for your domain
   - Subclasses for categories
   - Specific exceptions for specific errors

2. CONTEXT IS KEY:
   - Store relevant data as attributes
   - Override __str__ for readable messages
   - Include recovery suggestions when helpful

3. ERROR CODES (optional):
   - Useful for logging and monitoring
   - Makes errors searchable
   - Helps with internationalization

4. EXCEPTION CHAINING:
   - Use 'from e' to preserve original error
   - Helps with debugging

5. SERIALIZATION:
   - Implement to_dict() for API/logging
   - Include all relevant context
""")